In [3]:
import sys
import polars as pl
import random, re
from scipy.stats import norm

In [6]:
# TODO also simulate different tails:
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342
tail_5prime = [d_ribose, oxygen, 2 * oxygen + phosphorus, oxygen]

In [15]:
masses = pl.read_csv("../workflow/resources/masses_all.tsv",separator="\t")#, sep="\t").set_index("nucleoside", drop=False)
masses

nucleoside,monoisotopic_mass
str,f64
"""A""",267.09675
"""G""",283.09167
"""C""",243.08552
"""U""",244.06954
"""1A""",281.1124
…,…
"""10G""",409.1597
"""102G""",425.1547
"""105G""",538.2023


In [18]:
nucleoside_re = re.compile(r"\d*[ACGU]")
true_sequence = nucleoside_re.findall("CG100GUU")
n_fragments = 10
print(true_sequence)

['C', 'G', '100G', 'U', 'U']


In [20]:
# helper functions
def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l + 1, len(true_sequence))
    return l, r

In [22]:
fragments = pl.from_records(
    [random_fragment() for _ in range(n_fragments)],
    schema=["left", "right"],
    orient="row",
)
fragments

left,right
i64,i64
3,5
3,4
1,2
3,5
4,5
3,4
2,4
3,5
1,5


In [24]:
true_fragment_masses = [
    sum(
        masses.filter(pl.col("nucleoside") == b)
        .select(pl.col("monoisotopic_mass"))
        .item()
        for b in true_sequence[fragment["left"] : fragment["right"]]
    )
    for fragment in fragments.iter_rows(named=True)
]
true_fragment_masses


[488.13908,
 244.06954,
 283.09167,
 488.13908,
 244.06954,
 244.06954,
 551.16124,
 488.13908,
 1078.32245,
 526.17719]

In [25]:
fragment_noise = norm.rvs(scale=0.5, size=n_fragments)
fragment_noise

array([ 0.21384524,  0.98987511,  0.25094978,  0.22532455,  0.37831053,
       -0.47103108,  1.18595101, -0.04891137,  0.02433569,  0.02448658])

In [27]:
observed_fragment_masses = [
    max(true_mass + noise, 0.0)
    for noise, true_mass in zip(fragment_noise, true_fragment_masses)
]
observed_fragment_masses

[np.float64(488.35292523765605),
 np.float64(245.0594151094396),
 np.float64(283.3426197787502),
 np.float64(488.3644045507289),
 np.float64(244.4478505260432),
 np.float64(243.5985089186728),
 np.float64(552.3471910143614),
 np.float64(488.090168634736),
 np.float64(1078.3467856926538),
 np.float64(526.2016765845528)]

In [32]:
fragments = fragments.with_columns(
    (pl.col("left") == 0).alias("is_start"),
    (pl.col("right") == len(true_sequence)).alias("is_end"),
    pl.Series(true_fragment_masses).alias("true_mass"),
    pl.Series(observed_fragment_masses).alias("observed_mass"),
)
fragments

left,right,is_start,is_end,true_mass,observed_mass
i64,i64,bool,bool,f64,f64
3,5,false,true,488.13908,488.352925
3,4,false,false,244.06954,245.059415
1,2,false,false,283.09167,283.34262
3,5,false,true,488.13908,488.364405
4,5,false,true,244.06954,244.447851
3,4,false,false,244.06954,243.598509
2,4,false,false,551.16124,552.347191
3,5,false,true,488.13908,488.090169
1,5,false,true,1078.32245,1078.346786


In [45]:
num_parts = 2
breakpoints = random.sample(range(1, len(true_sequence)), num_parts - 1)
breakpoints

[3]

In [37]:
true_sequence

['C', 'G', '100G', 'U', 'U']

In [115]:
def random_fragment(num_parts = num_parts):
    if num_parts == 1:
        breakagepoints = [int(0)]
    else:
        breakagepoints = sorted(random.sample(range(1, len(true_sequence)), num_parts - 1))
        #Implement the condition that the num_parts cannot be greater than the sequence length!
    return breakagepoints

In [116]:
[random_fragment(num_parts = random.randint(1,3)) for _ in range(n_fragments)]

[[0], [0], [0], [0], [3], [4], [2, 3], [1, 2], [0], [2, 3]]

In [84]:
[random.randint(2,2) for _ in range(n_fragments)]

[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

In [118]:
breakagepoints = [random_fragment(num_parts = random.randint(1,3)) for _ in range(n_fragments)]
print(breakagepoints)

[[4], [0], [3], [1, 4], [1, 3], [1, 3], [4], [0], [1, 3], [1]]


In [176]:
def generate_left_right(breakagepoints):
    left,right = [],[]
    for exp in breakagepoints:
        left.append(0)
        for part in exp:
            if part != 0:
                right.append(part-1)
                left.append(part)
        right.append(len(true_sequence)-1)
    return left,right
l,r = generate_left_right(breakagepoints)
print(l)
print(r)

[0, 4, 0, 0, 3, 0, 1, 4, 0, 1, 3, 0, 1, 3, 0, 4, 0, 0, 1, 3, 0, 1]
[3, 4, 4, 2, 4, 0, 3, 4, 0, 2, 4, 0, 2, 4, 3, 4, 4, 0, 2, 4, 0, 4]


In [188]:
l_breakage,r_breakage = generate_left_right(breakagepoints) 

# sample random fragments from true sequence
fragments = pl.from_records(
    list(zip(l_breakage,r_breakage)), 
    schema=["left", "right"],
    orient="row",
)
fragments

left,right
i64,i64
0,3
4,4
0,4
0,2
3,4
…,…
0,0
1,2
3,4


In [198]:
masses = pl.read_csv("../workflow/resources/masses_all.tsv", separator="\t")#.set_index("nucleoside", drop=False)

In [199]:
masses

nucleoside,monoisotopic_mass
str,f64
"""A""",267.09675
"""G""",283.09167
"""C""",243.08552
"""U""",244.06954
"""1A""",281.1124
…,…
"""10G""",409.1597
"""102G""",425.1547
"""105G""",538.2023


In [202]:
true_fragment_masses = [
    sum(
        masses.filter(pl.col("nucleoside") == b)
        .select(pl.col("monoisotopic_mass"))
        .item()
        for b in true_sequence[fragment["left"] : fragment["right"] + 1]
    )
    for fragment in fragments.iter_rows(named=True)
]
true_fragment_masses

[1077.33843,
 244.06954,
 1321.40797,
 833.26889,
 488.13908,
 243.08552,
 834.2529099999999,
 244.06954,
 243.08552,
 590.18337,
 488.13908,
 243.08552,
 590.18337,
 488.13908,
 1077.33843,
 244.06954,
 1321.40797,
 243.08552,
 590.18337,
 488.13908,
 243.08552,
 1078.32245]

In [206]:
fragment_noise = norm.rvs(scale=0.5, size=len(true_fragment_masses))
observed_fragment_masses = [
    max(true_mass + noise, 0.0)
    for noise, true_mass in zip(fragment_noise, true_fragment_masses)
]
observed_fragment_masses

[np.float64(1077.841758723866),
 np.float64(243.862628301077),
 np.float64(1321.4848804971411),
 np.float64(833.6068379409538),
 np.float64(488.86297315968983),
 np.float64(243.28987402748135),
 np.float64(833.7682694443158),
 np.float64(244.68106449347934),
 np.float64(243.9708025231497),
 np.float64(590.0286820594827),
 np.float64(488.42035567772655),
 np.float64(242.87470715579988),
 np.float64(590.2245504801557),
 np.float64(488.6767642891667),
 np.float64(1077.6115076299168),
 np.float64(243.54411881208677),
 np.float64(1320.4671665636643),
 np.float64(243.25163640601843),
 np.float64(589.9374466155555),
 np.float64(487.4887652856656),
 np.float64(242.5098622415963),
 np.float64(1078.9379095316303)]

In [209]:
fragments = fragments.with_columns(
    (pl.col("left") == 0).alias("is_start"),
    (pl.col("right") == (len(true_sequence))-1).alias("is_end"),
    (pl.col("right") == pl.col("left")).alias("single_nucleotide"),
    pl.Series(true_fragment_masses).alias("true_mass"),
    pl.Series(observed_fragment_masses).alias("observed_mass"),
)
fragments


left,right,is_start,is_end,true_mass,observed_mass,single_nucleotide
i64,i64,bool,bool,f64,f64,bool
0,3,true,false,1077.33843,1077.841759,false
4,4,false,true,244.06954,243.862628,true
0,4,true,true,1321.40797,1321.48488,false
0,2,true,false,833.26889,833.606838,false
3,4,false,true,488.13908,488.862973,false
…,…,…,…,…,…,…
0,0,true,false,243.08552,243.251636,true
1,2,false,false,590.18337,589.937447,false
3,4,false,true,488.13908,487.488765,false
